This notebook reads PDFs directly with LLMs, using a **two-step** pipeline:
document **classification** and field **extraction** are performed in
*separate* LLM calls (unlike `pdf_to_llm.ipynb`, which does both in a single
call).

Pipeline: segment PDF → split → **classify each document** → deterministic gate
→ **extract invoice fields** only for invoice-like Originals.

In [2]:
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage
from dotenv import load_dotenv
from llm_maker import make_llm

load_dotenv()
llm = make_llm(model="pixtral-large", temperature=0.0)

In [7]:
from typing import Generic, TypeVar, Optional, Literal, List
from pydantic import BaseModel, Field, model_validator

T = TypeVar("T")

class Confident(BaseModel, Generic[T]):
    """A field value paired with the model's self-reported confidence and evidence.

    - value:      the extracted value (typed T)
    - confidence: 0..1 self-reported certainty this value is correct given the source
    - evidence:   verbatim snippet from the source text supporting the value
                  (used to verify the value is grounded, not hallucinated)
    """
    value: T
    confidence: float = Field(
        ge=0.0, le=1.0,
        description="Your certainty (0-1) that `value` is correct given ONLY the source text.",
    )
    evidence: str = Field(
        description="Short verbatim snippet copied from the source text that supports `value`.",
    )

In [4]:
# ---------------------------------------------------------------------------
# Node 1 — Document Segmentation
#
# These models are used ONLY to identify the boundaries of the independent
# accounting documents contained in a single PDF. They do NOT extract any
# invoice information. Extraction (ClassifyExtract) happens later, once per
# split document.
#
# IMPORTANT: the total page count is NOT part of this schema. A VLM cannot
# reliably count PDF pages, so the count is computed deterministically with
# pypdf and passed INTO the prompt. The model only assigns boundaries within
# that known 1..total_pages range.
# ---------------------------------------------------------------------------


class DocumentBoundary(BaseModel):
    """
    Boundary of a single independent accounting document inside a PDF.

    Pages are 1-based and inclusive on both ends.
    """

    start_page: int = Field(
        description="1-based page where this document starts (inclusive)."
    )
    end_page: int = Field(
        description="1-based page where this document ends (inclusive)."
    )
    confidence: float = Field(
        ge=0.0,
        le=1.0,
        description="Certainty (0-1) that these boundaries are correct.",
    )


class DocumentSegmentation(BaseModel):
    """
    Segmentation result: the independent accounting documents found in the
    PDF, in reading order.

    The total page count is known deterministically and is NOT part of this
    schema — the model only returns boundaries.
    """

    documents: list[DocumentBoundary] = Field(
        description="Independent accounting documents contained in the PDF, in page order."
    )


In [5]:


# class DocumentClassification(BaseModel):
#     """
#     Semantic classification of the document.

#     The goal is to identify the accounting nature of the document, independently
#     from its visual appearance.
#     """

#     document_type: Confident[
#         Literal[
#             "invoice",
#             "receipt",
#             "credit_note",
#             "debit_note",
#             "other"
#         ]
#     ] = Field(
#         description="""
#     Primary accounting document type.

#     Choose:

#     - invoice
#     - receipt
#     - credit_note
#     - debit_note
#     - other

#     Use 'other' whenever the document is not one of the above, even if it contains
#     financial information (shipping documents, customs documents, bank statements,
#     insurance certificates, purchase orders, etc.).

#     Do not infer the type from filenames.
#     Only use the document contents.
#     """
#     )

#     document_state: Optional[
#         Confident[
#             Literal[
#                 "original",
#                 "proforma",
#                 "copy",
#                 "cancelled"
#             ]
#         ]
#     ] = Field(
#         default=None,
#         description="""
#     Legal status of the document.

#     Examples:

#     Proforma Invoice (provisional invoice)
#     Original
#     Cópia
#     Duplicate (copy)
#     Duplicado (copy)
#     ANULADO (cancelled)
#     Cancelled (cancelled)

#     Return null when no state is explicitly indicated.
#     """
#     )

In [ ]:


class DocumentClassification(BaseModel):
    """
    Semantic classification of the document.

    The goal is to identify the accounting nature of the document, independently
    from its visual appearance.
    """

    document_type: Confident[
        Literal[
            "invoice",
            "receipt",
            "credit_note",
            "debit_note",
            "other"
        ]
    ] = Field(
        description="""
    Primary accounting document type.

    Choose:

    - invoice
    - receipt
    - credit_note
    - debit_note
    - other

    Use 'other' whenever the document is not one of the above, even if it contains
    financial information (shipping documents, customs documents, bank statements,
    insurance certificates, purchase orders, etc.).

    Do not infer the type from filenames.
    Only use the document contents.
    """
    )

    document_state: Optional[
        Confident[
            Literal[
                "original",
                "proforma",
                "copy",
                "cancelled"
            ]
        ]
    ] = Field(
        default=None,
        description="""
    Legal status of the document.

    Examples:

    Proforma Invoice (provisional invoice)
    Original
    Cópia
    Duplicate (copy)
    Duplicado (copy)
    ANULADO (cancelled)
    Cancelled (cancelled)

    Return null when no state is explicitly indicated.
    """
    )

In [56]:
class SemanticBlock(BaseModel):
    """
    A semantically meaningful block of text extracted from the document.

    A block should correspond to one logical section of the document
    (for example the invoice totals, VAT summary or line-item table),
    rather than an arbitrary OCR chunk.

    These blocks are intended for downstream deterministic parsing and
    validation. They should preserve the textual structure of the original
    document as much as possible.
    """

    kind: Literal[
        "all_parsed_text",
        "supplier_info",
        "client_info",
        "header",
        "line_items",
        "vat_summary",
        "totals",
        "footer",
        "other",
    ] = Field(
        description="""
    Semantic type of this text block.

    Choose the best matching category:

    - all_parsed_text:
        The entire text of the document, in reading order.
        Preserve all line breaks and spacing.

    - supplier_info:
        Supplier information as a formatted string.
        Preserve all line breaks and spacing.
    
    - client_info:
        Customer information as a formatted string.
        Preserve all line breaks and spacing.

    - header:
        Supplier/customer information, invoice number,
        invoice date, document identifiers, payment terms,
        document title, addresses.

    - line_items:
        The commercial table describing products or services.
        Preserve every row exactly as shown whenever possible.

    - vat_summary:
        VAT / tax breakdown tables.
        Includes taxable bases, VAT percentages,
        exemption reasons and VAT totals.

    - totals:
        Final monetary summary.
        Includes subtotal, discounts, VAT total,
        grand total, amount payable,
        currency and similar summary fields.

    - footer:
        Legal notes, payment instructions,
        bank information, disclaimers,
        QR references, certification statements.

    - other:
        Any meaningful block that does not fit the
        previous categories.
    """
    )

    page: int = Field(
        description="""
    1-based page number on which this block appears.
    """
        )

    text: str = Field(
        description="""
    Text contained in this semantic block.

    Guidelines:

    • Preserve line breaks whenever they help retain table structure.

    • Preserve row ordering.

    • Preserve negative signs.

    • Preserve decimal values exactly.

    • Do NOT rewrite, summarize or normalize.

    • Exclude unrelated neighbouring sections.

    Examples of a line_items block:

    Gasóleo Simples
    54,99   1,5878   81,95   23%   18,85
    56,04   1,5187   79,65   23%   18,32

    or

    REMUNERATION EXPORT CARGO
    2     52,00     104,00

    Examples of a totals block:

    Subtotal      507,22
    VAT            55,99
    Total         563,21
"""
    )

In [8]:
class InvoiceData(BaseModel):
    """
    Core accounting information extracted from an invoice-like document.

    Every field is optional.

    Never guess.
    Prefer null whenever multiple values could match.
    """

    supplier_name: Optional[Confident[str]] = Field(
        default=None,
        description="""
        Legal or commercial name of the company issuing the invoice.

        Usually located:

        - top header
        - company logo
        - company address block

        This is the seller.

        Do NOT return:

        - customer
        - recipient
        - ship-to
        - consignee
        - billing contact
        """
    )

    supplier_vat: Optional[Confident[str]] = Field(
        default=None,
        description="""
        Tax identification number (VAT/NIF) of the supplier.

        Usually appears close to the supplier name.

        Examples:

        PT501925350
        PT500697370
        NL800822274B01

        For Portuguese invoices:

        If exactly nine digits are shown without country prefix,
        normalize as:

        PT#########

        Do not invent prefixes for other countries.
        """
    )


    client_name: Optional[Confident[str]] = Field(
        default=None,
        description="""
        Company or person receiving the invoice.

        Usually located in:

        - customer block
        - invoice address
        - 'Exmo(s). Senhor(es)'
        - Bill To
        - Customer

        Do not confuse with supplier.
        """
    )

    client_vat: Optional[Confident[str]] = Field(
        default=None,
        description="""
        VAT number of the customer.

        Usually appears close to the customer name.

        Normalize Portuguese NIFs to PT######### when only nine digits
        are shown.

        Return null if no customer VAT appears.
        """
    )

    purchase_order: List[Confident[str]] = Field(
        default_factory=list,
        description="""
        All purchase order numbers or references found in the invoice.
        Usually appear in the header or customer block.
        Return an empty list if none are present.
        Purchase orders have 10 digits
        """
    )

    issue_date: Optional[Confident[str]] = Field(
        default=None,
        description="""
        Invoice issue date.

        Normalize to:

        DD-MM-YYYY

        Look for labels such as:

        Issue Date
        Data
        Data de emissão
        Invoice Date

        Do NOT use:

        Due Date
        Shipment Date
        Check-in
        Check-out
        Operation Date
        Service Period
        """
    )

    base_amount: Optional[Confident[float]] = Field(
        default=None,
        description="""
        Net amount before VAT.

        Usually labelled:

        Base
        Valor Líquido
        Net Amount
        Taxable Amount
        Subtotal

        Do not return the total including VAT.

        When VAT is zero due to exemption or reverse charge,
        this value may equal the total amount.
        """
    )

    vat_amount: Optional[Confident[float]] = Field(
        default=None,
        description="""
        Total VAT amount charged on the invoice.

        This is an absolute monetary amount.

        Examples:

        55.99
        189.60
        0.00

        Do NOT return:

        23
        16
        6

        Those are VAT rates, not VAT amounts.
        """
    )

    total_amount: Optional[Confident[float]] = Field(
        default=None,
        description="""
        Final payable amount including VAT.

        Usually labelled:

        Total
        Total Invoice
        Valor Total
        Total da Fatura
        Amount Due

        For zero-VAT invoices:

        total = base amount.

        Do not return intermediate subtotals.
        """
    )

    currency: Optional[Confident[str]] = Field(
        default=None,
        description="""
        Currency code of the invoice.

        Use ISO 4217 three-letter codes:
        EUR, USD, GBP, JPY, etc.
        If no currency is explicitly indicated, return null.
        """
    )

    @model_validator(mode="before")
    @classmethod
    def drop_empty_fields(cls, data):
        """Coerce degenerate Confident objects to None.

        The extraction prompt asks the model to return null for fields it
        cannot find, but the model sometimes signals "unknown" by returning a
        Confident object with no `value` (evidence="", confidence=0) instead.
        `Confident.value` is required, so that shape raises a ValidationError
        and loses the whole document. Normalize it to the null the schema
        expects, keeping the prompt instruction as a hint rather than a
        load-bearing constraint.
        """
        if isinstance(data, dict):
            return {
                k: (
                    None
                    if isinstance(v, dict) and v.get("value") in (None, "")
                    else v
                )
                for k, v in data.items()
            }
        return data

#     semantic_blocks: Optional[list[SemanticBlock]] = Field(
#         default=None,
#         description="""
#         Relevant semantic sections extracted from the document.

#         These blocks are intended to support downstream validation and
#         table parsing without requiring another Vision LLM call.

#         Include all possibly useful blocks for accounting validation.

#         Typical documents contain between two and five blocks:

#         • Whole parsed text
#         • Header
#         • Line items
#         • VAT summary
#         • Totals

#         Do not include repeated pages, advertisements,
#         or unrelated attachments unless they contain information
#         needed to validate the extracted invoice.

#         If a section does not exist (for example an invoice without
#         a VAT summary), simply omit that block.
#         """
# )

    # parsed_text: Optional[str] = Field(
    #     default=None,
    #     description="""
    #     The entire text of the document, in reading order.
    #     Preserve all line breaks and spacing.
    #     If there are tables, preserve their row ordering and spacing as much as possible.
    #     """
    # )

In [58]:
# ---------------------------------------------------------------------------
# Two-step pipeline: classification and extraction are SEPARATE LLM calls.
#
# Unlike pdf_to_llm.ipynb (which used a single combined `ClassifyExtract`
# schema), here we keep the per-step schemas independent:
#
#   • Step 3a — classification  -> DocumentClassification   (defined above)
#   • Step 3b — extraction       -> InvoiceData              (defined above)
#
# There is intentionally NO combined `ClassifyExtract` model. The two steps
# are wired together in Python, with a deterministic gate deciding whether the
# extraction call runs at all.
# ---------------------------------------------------------------------------

In [9]:
import base64
import io
from pathlib import Path

from pypdf import PdfReader

#Invoices to use in testing
# attachment_path = Path("docs/2150768990.pdf")
# attachment_path = Path("docs/149-2026 CA Indosuez Portugal_Lumiso SGPS Unipessoal LDA_Custody Commission excluding tax Q2 2026.pdf")  
# attachment_path = Path("docs/20260625185316.pdf")  
# attachment_path = Path("docs/FT-4302108.pdf")  
# attachment_path = Path("docs/8940.CCDO.10443.2026.pdf")
attachment_path = Path("docs/Invoice 229555.pdf")

print(attachment_path)

with open(attachment_path, "rb") as f:
    pdf_bytes = f.read()
    encoded_pdf = base64.b64encode(pdf_bytes).decode("utf-8")

# Page count is deterministic — never ask the VLM to count pages.
total_pages = len(PdfReader(io.BytesIO(pdf_bytes)).pages)
print(f"Total pages (deterministic): {total_pages}")


docs\Invoice 229555.pdf
Total pages (deterministic): 1


In [10]:
# ---------------------------------------------------------------------------
# Node 1 — Run Document Segmentation
#
# The VLM cannot count pages reliably, so we DON'T ask it to. We pass the
# deterministic `total_pages` into the prompt and the model only assigns
# boundaries within the known 1..total_pages range.
# ---------------------------------------------------------------------------

segmentation_system_message = SystemMessage(
    content=f"""
You are a document segmentation system for accounting documents.

Your task is ONLY to identify the boundaries of the independent accounting
documents inside the attached PDF.

The PDF has exactly {total_pages} page(s). This number is authoritative — do
NOT re-count the pages and do NOT return page numbers outside the range
1..{total_pages}.

For each document return:

- start_page (1-based, inclusive, between 1 and {total_pages})
- end_page   (1-based, inclusive, between 1 and {total_pages})
- confidence (0-1)

Do NOT extract invoice information (no supplier, amounts, VAT, dates, etc.).

A new document begins whenever a new invoice, receipt, credit note, debit
note or unrelated accounting document starts. 

There can be multiple independent documents in a single PDF, and they may be
of different types.

There can also be multiple copies of the same document. Treat each copy as a
DISTINCT document and return it separately. If there is an original and copy, 
there should be two SEPERATE entries in the output.

Cover every page exactly once: the boundaries must be contiguous and
non-overlapping. The first document must start at page 1 and the last document
must end at page {total_pages}.
"""
)

segmentation_human_message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": (
                f"This PDF has {total_pages} page(s). "
                "Identify the boundaries of the independent accounting documents. "
                f"All page numbers must be between 1 and {total_pages}."
            ),
        },
        {
            "type": "file",
            "file": {
                "file_data": f"data:application/pdf;base64,{encoded_pdf}",
                "filename": attachment_path.name,
                "format": "application/pdf",
            },
        },
    ]
)

segmentation_llm = llm.with_structured_output(DocumentSegmentation)

segmentation = segmentation_llm.invoke([
    segmentation_system_message,
    segmentation_human_message,
])


def validate_segmentation(documents: list[DocumentBoundary], total_pages: int) -> list[str]:
    """Check the VLM boundaries against the deterministic page count.

    Returns a list of human-readable problems (empty == clean coverage).
    """
    problems: list[str] = []
    if not documents:
        return [f"No documents returned for a {total_pages}-page PDF."]

    ordered = sorted(documents, key=lambda d: d.start_page)
    covered: set[int] = set()
    expected_next = 1
    for d in ordered:
        if not (1 <= d.start_page <= total_pages) or not (1 <= d.end_page <= total_pages):
            problems.append(
                f"Out-of-range boundary {d.start_page}-{d.end_page} (valid 1-{total_pages})."
            )
        if d.end_page < d.start_page:
            problems.append(f"Inverted boundary {d.start_page}-{d.end_page}.")
        if d.start_page != expected_next:
            problems.append(
                f"Gap/overlap: expected next document to start at page {expected_next}, "
                f"got {d.start_page}."
            )
        for p in range(max(d.start_page, 1), min(d.end_page, total_pages) + 1):
            if p in covered:
                problems.append(f"Page {p} covered by more than one document.")
            covered.add(p)
        expected_next = d.end_page + 1

    missing = set(range(1, total_pages + 1)) - covered
    if missing:
        problems.append(f"Pages not covered by any document: {sorted(missing)}.")
    return problems


segmentation_problems = validate_segmentation(segmentation.documents, total_pages)

print("Detected documents")
print("------------------")
print(f"Total pages: {total_pages}")
for i, doc in enumerate(segmentation.documents, start=1):
    print(
        f"{i}. pages {doc.start_page}-{doc.end_page}  "
        f"confidence={doc.confidence:.2f}"
    )

if segmentation_problems:
    print("\n⚠️  Segmentation coverage issues:")
    for problem in segmentation_problems:
        print(f"  - {problem}")
else:
    print("\n✅ Boundaries cover all pages exactly once.")


Detected documents
------------------
Total pages: 1
1. pages 1-1  confidence=1.00

✅ Boundaries cover all pages exactly once.


In [11]:
# ---------------------------------------------------------------------------
# Node 2 — Split PDF
#
# If the segmentation found a single document, do nothing (reuse the original
# PDF). Otherwise, split the source PDF into one child PDF per detected
# document, keeping the original page bytes.
#
# Each entry in `split_documents` is a self-contained unit for the extraction
# node: filename + base64 payload + the boundary that produced it.
# ---------------------------------------------------------------------------

from dataclasses import dataclass

from pypdf import PdfWriter


@dataclass
class SplitDocument:
    filename: str
    encoded_pdf: str
    boundary: DocumentBoundary


def split_pdf(pdf_bytes: bytes, base_filename: str, boundaries: list[DocumentBoundary]) -> list[SplitDocument]:
    stem = Path(base_filename).stem
    suffix = Path(base_filename).suffix or ".pdf"

    # Single document -> no split, reuse the original bytes as-is.
    if len(boundaries) <= 1:
        boundary = boundaries[0] if boundaries else None
        return [
            SplitDocument(
                filename=base_filename,
                encoded_pdf=base64.b64encode(pdf_bytes).decode("utf-8"),
                boundary=boundary,
            )
        ]

    reader = PdfReader(io.BytesIO(pdf_bytes))
    page_count = len(reader.pages)

    split_documents: list[SplitDocument] = []
    for i, boundary in enumerate(boundaries, start=1):
        writer = PdfWriter()
        # start_page/end_page are 1-based inclusive; PdfReader.pages is 0-based.
        # Clamp to the real page count as a safety net against bad boundaries.
        start_idx = max(boundary.start_page - 1, 0)
        end_idx = min(boundary.end_page, page_count)
        for page_idx in range(start_idx, end_idx):
            writer.add_page(reader.pages[page_idx])

        buffer = io.BytesIO()
        writer.write(buffer)
        child_bytes = buffer.getvalue()

        split_documents.append(
            SplitDocument(
                filename=f"{stem}_{i:03d}{suffix}",
                encoded_pdf=base64.b64encode(child_bytes).decode("utf-8"),
                boundary=boundary,
            )
        )

    return split_documents


split_documents = split_pdf(pdf_bytes, attachment_path.name, segmentation.documents)

print(f"{len(split_documents)} document(s) ready for extraction:")
for doc in split_documents:
    b = doc.boundary
    print(f"  {doc.filename}  (pages {b.start_page}-{b.end_page})" if b else f"  {doc.filename}")


1 document(s) ready for extraction:
  Invoice 229555.pdf  (pages 1-1)


In [12]:
# ---------------------------------------------------------------------------
# Node 2b — Persist the split documents to disk
#
# `split_pdf` keeps every child PDF in memory (base64 only). The deterministic
# parser (`build_attachment_evidence`) works off a Path, so write each split to
# docs/splits/<original-stem>/ and keep a filename -> Path map.
# ---------------------------------------------------------------------------

split_dir = Path("docs/splits") / attachment_path.stem
split_dir.mkdir(parents=True, exist_ok=True)

split_paths: dict[str, Path] = {}
for doc in split_documents:
    path = split_dir / doc.filename
    path.write_bytes(base64.b64decode(doc.encoded_pdf))
    split_paths[doc.filename] = path

print(f"{len(split_paths)} split document(s) written to {split_dir}:")
for filename, path in split_paths.items():
    print(f"  {path}  ({path.stat().st_size:,} bytes)")

1 split document(s) written to docs\splits\Invoice 229555:
  docs\splits\Invoice 229555\Invoice 229555.pdf  (120,136 bytes)


In [13]:
# ---------------------------------------------------------------------------
# Step 3a — Classification prompt & message builder
#
# Classification-only: the model returns just DocumentClassification (type +
# state). No supplier/amount/VAT extraction and no semantic blocks here — that
# is deferred to the separate extraction step (3b).
# ---------------------------------------------------------------------------


def build_classification_human_message(filename: str, encoded_pdf: str) -> HumanMessage:
    """Build the classification HumanMessage for a single (already segmented) document."""
    return HumanMessage(
        content=[
            {
                "type": "text",
                "text": """
Analyze the attached PDF.

Classify ONLY the accounting nature of the document (document_type) and its
legal state (document_state) when explicitly visible.

Do NOT extract any invoice fields (no supplier, client, amounts, VAT, dates).

Follow the provided schema exactly.
""",
            },
            {
                "type": "file",
                "file": {
                    "file_data": f"data:application/pdf;base64,{encoded_pdf}",
                    "filename": filename,
                    "format": "application/pdf",
                },
            },
        ]
    )


classification_system_message = SystemMessage(
    content="""
You are an expert document understanding system specialized in invoices and accounting documents.

Your task is ONLY to classify the document into the provided schema. You must
NOT extract invoice fields — a separate step handles extraction.

GENERAL RULES

- The schema is the source of truth. Follow every field description exactly.
- If a value cannot be determined confidently from the document, return null.
- Never invent, estimate or complete missing information.
- Prefer returning null rather than guessing.
- Use only information present in the document.
- Ignore filenames, metadata and external knowledge.

CLASSIFICATION

Determine the accounting nature of the document (document_type):

- invoice
- receipt
- credit_note
- debit_note
- other

Use 'other' whenever the document is not one of the above, even if it contains
financial information (shipping documents, customs documents, bank statements,
insurance certificates, purchase orders, etc.).

Also determine the document state (document_state) ONLY when explicitly visible:

- original
- proforma
- copy
- cancelled

Return null for document_state when no state is explicitly indicated.

CONFIDENCE

Every classified field must include an honest confidence score between 0 and 1.

1.00
    Explicitly visible and completely unambiguous.

0.90-0.99
    Clearly visible with only minor formatting or OCR uncertainty.

0.70-0.89
    Probably correct but there is some ambiguity.

0.50-0.69
    Weak evidence or partially inferred.

Below 0.50
    The value is too uncertain. Prefer returning null instead.

EVIDENCE

Every classified value must include evidence.

Evidence must:

- be copied VERBATIM from the document;
- never be paraphrased;
- be as short as possible while uniquely supporting the value;
- preserve original spelling, punctuation and formatting whenever possible.

GOOD:
    "Fatura"
    "Proforma Invoice"
    "DUPLICADO"

BAD:
    "This is an invoice."
    "The document is a copy."

FINAL RULE

Accuracy is more important than completeness. It is always preferable to return
a lower-confidence classification honestly than to guess.
"""
)


In [ ]:
# ---------------------------------------------------------------------------
# Step 3a — Classification (one LLM call per split document)
#
# Robustness notes (same rationale as the segmentation node):
# - pixtral-large via Bedrock REJECTS tool_choice / parallel_tool_calls, so we
#   stay on the default structured-output path (no method="function_calling").
# - If the model replies with prose instead of JSON we catch the parse error
#   per-document and retry ONCE with a stricter "JSON only" nudge, so one bad
#   document neither crashes the batch nor is silently dropped.
# ---------------------------------------------------------------------------

from langchain_core.exceptions import OutputParserException
from pydantic import ValidationError

# Parse failures we want to retry (prose instead of JSON), as opposed to
# transport/HTTP errors which won't improve on a plain retry.
PARSE_ERRORS = (ValidationError, OutputParserException, ValueError)

json_only_nudge = HumanMessage(
    content=(
        "Your previous reply could not be parsed. Respond again with ONLY a single "
        "JSON object that matches the required schema — no prose, no Markdown, no "
        "code fences."
    )
)

classify_llm = llm.with_structured_output(DocumentClassification)


def classify_document(doc: SplitDocument) -> DocumentClassification:
    human_message = build_classification_human_message(doc.filename, doc.encoded_pdf)
    try:
        return classify_llm.invoke([classification_system_message, human_message])
    except PARSE_ERRORS:
        # One retry with an explicit JSON-only reminder appended.
        return classify_llm.invoke(
            [classification_system_message, human_message, json_only_nudge]
        )


classifications: list[tuple[SplitDocument, DocumentClassification]] = []
classify_failures: list[tuple[SplitDocument, Exception]] = []

for doc in split_documents:
    try:
        classification = classify_document(doc)
    except Exception as exc:  # noqa: BLE001 - keep the batch alive, inspect after
        classify_failures.append((doc, exc))
        print(f"❌ Failed to classify {doc.filename}: {type(exc).__name__}: {exc}")
        continue

    classifications.append((doc, classification))
    print(
        f"✅ Classified {doc.filename}: "
        f"{classification.document_type.value}"
        + (
            f" / {classification.document_state.value}"
            if classification.document_state is not None
            else " / (no state)"
        )
    )

print(
    f"\n{len(classifications)} classified, {len(classify_failures)} failed "
    f"out of {len(split_documents)} document(s)."
)

✅ Classified Invoice 229555.pdf: other

1 classified, 0 failed out of 1 document(s).


In [ ]:

# ---------------------------------------------------------------------------
# Step 3b — Extraction prompt & message builder
#
# Extraction-only: the model returns just InvoiceData. The document has ALREADY
# been classified as an invoice-like Original by step 3a; the confirmed
# classification is injected as context so the model does not re-classify.
# ---------------------------------------------------------------------------


def build_extraction_human_message(
    filename: str,
    encoded_pdf: str,
    classification: "DocumentClassification",
) -> HumanMessage:
    """Build the extraction HumanMessage for a single, already-classified document.

    The confirmed classification (type + state) is passed as known context so
    the model focuses on extracting InvoiceData rather than re-classifying.
    """
    doc_type = classification.document_type.value
    doc_state = (
        classification.document_state.value
        if classification.document_state is not None
        else "original (not explicitly stated)"
    )
    return HumanMessage(
        content=[
            {
                "type": "text",
                "text": f"""
Analyze the attached PDF.

This document has ALREADY been classified as:
  - document_type:  {doc_type}
  - document_state: {doc_state}

Do NOT re-classify it. Extract the requested structured invoice information
into the provided schema, following every field description exactly.
""",
            },
            {
                "type": "file",
                "file": {
                    "file_data": f"data:application/pdf;base64,{encoded_pdf}",
                    "filename": filename,
                    "format": "application/pdf",
                },
            },
        ]
    )


extraction_system_message = SystemMessage(
    content="""
You are an expert document understanding system specialized in invoices and accounting documents.

The document has ALREADY been classified as an invoice-like Original by a
previous step. Your task is ONLY to extract the structured information into the
provided InvoiceData schema. Do NOT re-classify the document.

GENERAL RULES

- The schema is the source of truth. Follow every field description exactly.
- Every field is optional.
- When a field is not present in the document, set the ENTIRE field to null. 
Do NOT return a Confident object with an empty value, empty evidence, 
or zero confidence — omit the whole field (return null for it) instead.- Never invent, estimate or complete missing information.
- Prefer returning null rather than guessing.
- Use only information present in the document.
- Ignore filenames, metadata and external knowledge.

CONFIDENCE

Every extracted field must include an honest confidence score between 0 and 1.

1.00
    Explicitly visible and completely unambiguous.

0.90-0.99
    Clearly visible with only minor formatting or OCR uncertainty.

0.70-0.89
    Probably correct but there is some ambiguity.

0.50-0.69
    Weak evidence or partially inferred.

Below 0.50
    The value is too uncertain. Prefer returning null instead.

EVIDENCE

Every extracted value must include evidence.

Evidence must:

- be copied VERBATIM from the document;
- never be paraphrased;
- be as short as possible while uniquely supporting the value;
- preserve original spelling, punctuation and formatting whenever possible.

GOOD:
    "Petrogal, S.A."
    "Total da Fatura 1.607,70"
    "VAT Number PT511034750"

BAD:
    "The supplier is Petrogal."
    "Invoice total."

NORMALIZATION

Normalize values only when requested by the schema.

Examples:

- Dates -> DD-MM-YYYY
- Monetary values -> numeric values only
- Portuguese NIFs:
    if exactly 9 digits are shown without country prefix,
    normalize to PT#########.

Do not normalize evidence.

FINAL RULE

Accuracy is more important than completeness.

It is always preferable to return fewer fields with high confidence than many uncertain fields.
"""
)

# SEMANTIC BLOCKS

# When semantic blocks are requested:

# - preserve the original reading order;
# - preserve table rows;
# - preserve line breaks when useful;
# - exclude advertisements, decorative elements, QR codes, bank details,
#   long legal disclaimers and unrelated attachments unless they are required
#   to validate the extracted accounting information.

In [ ]:
# ---------------------------------------------------------------------------
# Step 3b — Deterministic gate + Extraction (one LLM call per gated-in document)
#
# The gate is pure Python: extraction runs ONLY for invoice-like Originals.
# A null document_state is treated as "original" (real documents rarely print
# the word "Original"); only an explicit copy / proforma / cancelled skips
# extraction. Non-invoice types (receipt, debit_note, other) are skipped too.
#
# `results` pairs every document with its classification and either the
# extracted InvoiceData or None (gated out / extraction failed).
# ---------------------------------------------------------------------------


def should_extract(cls: DocumentClassification) -> bool:
    # Null state is treated as "original" (extract). Only an explicit
    # copy / proforma / cancelled skips extraction.
    return (
        cls.document_type.value in ("invoice", "credit_note")
        and (cls.document_state is None or cls.document_state.value == "original")
    )


extract_llm = llm.with_structured_output(InvoiceData)


def extract_document(doc: SplitDocument, cls: DocumentClassification) -> InvoiceData:
    human_message = build_extraction_human_message(doc.filename, doc.encoded_pdf, cls)
    try:
        return extract_llm.invoke([extraction_system_message, human_message])
    except PARSE_ERRORS:
        # One retry with an explicit JSON-only reminder appended.
        return extract_llm.invoke(
            [extraction_system_message, human_message, json_only_nudge]
        )


results: list[tuple[SplitDocument, DocumentClassification, Optional[InvoiceData]]] = []
extract_failures: list[tuple[SplitDocument, Exception]] = []

for doc, cls in classifications:
    if not should_extract(cls):
        state = cls.document_state.value if cls.document_state is not None else "no state"
        print(f"⏭️  Skipped {doc.filename}: classified as {cls.document_type.value} / {state}")
        results.append((doc, cls, None))
        continue

    try:
        invoice_data = extract_document(doc, cls)
    except Exception as exc:  # noqa: BLE001 - keep the batch alive, inspect after
        extract_failures.append((doc, exc))
        print(f"❌ Failed to extract {doc.filename}: {type(exc).__name__}: {exc}")
        results.append((doc, cls, None))
        continue

    results.append((doc, cls, invoice_data))
    print(f"✅ Extracted {doc.filename}")

extracted_count = sum(1 for _, _, data in results if data is not None)
print(
    f"\n{extracted_count} extracted, {len(extract_failures)} failed, "
    f"{len(classifications) - extracted_count - len(extract_failures)} gated out "
    f"of {len(classifications)} classified document(s)."
)

⏭️  Skipped Invoice 229555.pdf: classified as other

0 extracted, 0 failed, 1 gated out of 1 classified document(s).


In [83]:
from typing import Any


def _confidence_icon(confidence: float | None) -> str:
    if confidence is None:
        return "⚪"
    if confidence >= 0.9:
        return "🟢"
    if confidence >= 0.7:
        return "🟡"
    return "🔴"


def _print_field(label: str, field: Any) -> None:
    """
    Pretty-print one Confident[T] field.

    Expected shape:
        field.value
        field.confidence
        field.evidence
    """
    if field is None:
        print(f"{label:<15} {'-':<35}")
        return

    value = str(field.value) if field.value is not None else "-"
    confidence = getattr(field, "confidence", None)

    print(
        f"{label:<15}"
        f"{value:<40}"
        f"{_confidence_icon(confidence)} "
        f"{confidence:.2f}"
    )

    evidence = getattr(field, "evidence", None)
    if evidence:
        print(f"{'':15}evidence: {evidence}")


def print_classification_result(filename: str, classification: DocumentClassification) -> None:
    print("=" * 80)
    print(f"📄 {filename}")
    print("=" * 80)

    print("\nClassification")
    print("-" * 14)
    _print_field("Type", classification.document_type)
    _print_field("State", classification.document_state)


def print_extraction_result(
    filename: str,
    classification: DocumentClassification,
    invoice: Optional[InvoiceData],
) -> None:
    print("=" * 80)
    print(f"📄 {filename}")
    print("=" * 80)

    if invoice is None:
        state = (
            classification.document_state.value
            if classification.document_state is not None
            else "no state"
        )
        print(f"\n⏭️  No extraction — classified as {classification.document_type.value} / {state}.")
        return

    print("\nParties")
    print("-" * 7)
    _print_field("Supplier", invoice.supplier_name)
    _print_field("Supplier VAT", invoice.supplier_vat)
    _print_field("Client", invoice.client_name)
    _print_field("Client VAT", invoice.client_vat)

    print("\nAmounts")
    print("-" * 7)
    _print_field("Issue date", invoice.issue_date)
    _print_field("Base", invoice.base_amount)
    _print_field("VAT", invoice.vat_amount)
    _print_field("Total", invoice.total_amount)


# --- Step 3a report: classification for every document -----------------------
print("#" * 80)
print("# CLASSIFICATION RESULTS")
print("#" * 80)
print()
for doc, cls, _ in results:
    print_classification_result(doc.filename, cls)
    print()

# --- Step 3b report: extraction (or skip reason) for every document ----------
print("#" * 80)
print("# EXTRACTION RESULTS")
print("#" * 80)
print()
for doc, cls, invoice_data in results:
    print_extraction_result(doc.filename, cls, invoice_data)
    print()


print("#" * 80)

################################################################################
# CLASSIFICATION RESULTS
################################################################################

📄 Invoice 229555.pdf

Classification
--------------
Type           invoice                                 🟢 1.00
               evidence: INVOICE
State          original                                🟢 0.90
               evidence: INVOICE

################################################################################
# EXTRACTION RESULTS
################################################################################

📄 Invoice 229555.pdf

Parties
-------
Supplier       GSLines Transportes Maritimos LDA       🟢 1.00
               evidence: GSLines Transportes Maritimos LDA
Supplier VAT   NL813715532B01                          🟢 1.00
               evidence: NL813715532B01
Client         RateBase Exchange Total EUR             🟡 0.80
               evidence: RateBase Exchange Total EUR
Client VA

In [84]:
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage

# Helper function to execute tool calls
def execute_tool_calls(response, tools_dict):
    """Execute tool calls and return results."""
    results = []
    for tool_call in response.tool_calls:
        tool = tools_dict.get(tool_call['name'])
        if tool:
            result = tool.invoke(tool_call['args'])
            results.append({
                'tool_call_id': tool_call['id'],
                'result': result
            })
    return results

def ask_with_tools(question, llm_with_tools, tools_list, system_message=None):
    """Ask a question and automatically handle tool calls.

    Args:
        question: a str, a BaseMessage, or a list of BaseMessages
        llm_with_tools: LLM with tools bound
        tools_list: List of tool objects
        system_message: optional SystemMessage prepended to the conversation

    Returns:
        Final answer as string
    """
    # Normalize `question` to a list of messages — `invoke` rejects a bare
    # BaseMessage (it accepts PromptValue, str, or list[BaseMessage]).
    if isinstance(question, str):
        messages = [HumanMessage(content=question)]
    elif isinstance(question, BaseMessage):
        messages = [question]
    else:
        messages = list(question)

    if system_message is not None:
        messages = [system_message, *messages]

    response = llm_with_tools.invoke(messages)

    if response.tool_calls:
        # Create tools dictionary
        tools_dict = {tool.name: tool for tool in tools_list}

        # Execute tools
        tool_results = execute_tool_calls(response, tools_dict)

        # Continue the SAME conversation: prior messages + the tool-calling
        # response + one ToolMessage per executed call.
        messages = [*messages, response]
        for tool_result in tool_results:
            messages.append(
                ToolMessage(
                    content=str(tool_result['result']),
                    tool_call_id=tool_result['tool_call_id']
                )
            )

        final_response = llm_with_tools.invoke(messages)
        return final_response.content
    else:
        return response.content

print("Helper function defined and ready to use!")

Helper function defined and ready to use!


In [85]:
@tool
def verify_client_nif(client_vat: str, supplier_vat: str) -> tuple[bool, bool]:
    """Verify wether a the nifs recovered are in the list of known clients.
    
    Args:
        client_vat: The unique client VAT number
        supplier_vat: The unique supplier VAT number
    
    Returns:
        True, True if the client and supplier vats are in the list of known clients, otherwise False
    """
    # Simulated database
    VAT_LIST = ['PT511034750', 'PT511011911', 'PT509225918', 'PT513286004', 'PT511051000', 'PT511070357', 'PT513791345']
    

    if client_vat in VAT_LIST:
        client = True
    else:
        client = False
    if supplier_vat in VAT_LIST:
        supplier = True
    else:
        supplier = False

    return client, supplier

@tool
def verify_supplier_nif(client_vat: str, supplier_vat: str) -> tuple[bool, bool]:
    """Verify wether a the nifs recovered are in the list of known suppliers.
    
    Args:
        client_vat: The unique client VAT number
        supplier_vat: The unique supplier VAT number
    
    Returns:
        True, True if the client and supplier are in the list of known suppliers, otherwise False
    """
    # Simulated database
    VAT_LIST = ['PT512046158', 'PT500697370', 'LU19578473', 'PT501925350', 'NL800822274B01']
    

    if client_vat in VAT_LIST:
        client = True
    else:
        client = False
    if supplier_vat in VAT_LIST:
        supplier = True
    else:
        supplier = False

    return client, supplier


In [86]:

def build_validation_human_message(
    extraction: "InvoiceData",
    parsed_text: Optional[str] = None,
) -> HumanMessage:
    """Build the validation message for a single, already-extracted document.
    It should include the extracted invoice data and the relevant semantic blocks for context, and ask the model to validate the data against the blocks.
    """

    extracted_data = {
        "supplier_name": extraction.supplier_name.value if extraction.supplier_name else None,
        "supplier_vat": extraction.supplier_vat.value if extraction.supplier_vat else None,
        "client_name": extraction.client_name.value if extraction.client_name else None,
        "client_vat": extraction.client_vat.value if extraction.client_vat else None,
        "purchase_order": extraction.purchase_order.value if extraction.purchase_order else None,
        "issue_date": extraction.issue_date.value if extraction.issue_date else None,
        "base_amount": extraction.base_amount.value if extraction.base_amount else None,
        "vat_amount": extraction.vat_amount.value if extraction.vat_amount else None,
        "total_amount": extraction.total_amount.value if extraction.total_amount else None,
        "currency": extraction.currency.value if extraction.currency else None,
    }

        
    return HumanMessage(
        content=[
            {
                "type": "text",
                "text": f"""Validate the extracted invoice data against the provided parsed text and any available validation tools. Some information 
                may have been extracted from images, and it may not be in the parsed text. Use the parsed text and any available tools to validate the extracted data.
                The extracted data is as follows:
                {extracted_data}
                The parsed text is as follows:
                {parsed_text if parsed_text else "No parsed text provided."}
                Do not assume the extracted values are correct. For every field, determine whether it is valid, invalid, or unable_to_validate. 
                When invalid, provide the corrected value only if it is explicitly supported by the semantic blocks. 
                Include supporting evidence and explain any discrepancies. Use the available validation tools whenever applicable.
                """
            }
        ]
    )


validation_system_message = SystemMessage(
    content="""
You are an expert document understanding system specialized in invoices and accounting documents.

The document has ALREADY been classified as an invoice-like Original by a
previous step. Your task is ONLY to extract the structured information into the
provided InvoiceData schema. Do NOT re-classify the document.

GENERAL RULES

- The schema is the source of truth. Follow every field description exactly.
- Every field is optional.
- When a field is not present in the document, set the ENTIRE field to null. 
Do NOT return a Confident object with an empty value, empty evidence, 
or zero confidence — omit the whole field (return null for it) instead.- Never invent, estimate or complete missing information.
- Prefer returning null rather than guessing.
- Use only information present in the document.
- Ignore filenames, metadata and external knowledge.

CONFIDENCE

Every extracted field must includes an honest confidence score between 0 and 1.


EVIDENCE - every extracted value includes evidence. Do not alter the evidence that is inputted to the model. 

GOOD:
    "Petrogal, S.A."
    "Total da Fatura 1.607,70"
    "VAT Number PT511034750"

BAD:
    "The supplier is Petrogal."
    "Invoice total."

NORMALIZATION

Normalize values only when requested by the schema.

Examples:

- Dates -> DD-MM-YYYY
- Monetary values -> numeric values only
- Portuguese NIFs:
    if exactly 9 digits are shown without country prefix,
    normalize to PT#########.

Do not normalize evidence.

TOOLS
You will have access to the following tools to validate the extracted data:
- verify_client_nif(client_vat: str, supplier_vat: str) -> tuple[bool, bool]: Verify whether the client and supplier NIFs are in the list of known clients. Returns True, True if both are in the list, otherwise False.
- verify_supplier_nif(client_vat: str, supplier_vat: str) -> tuple[bool, bool]: Verify whether the client and supplier NIFs are in the list of known suppliers. Returns True, True if both are in the list, otherwise False.

Use the tools to validate the supplier and client exist. 
If client appears in supplier list, or supplier appears in client list, we should most likely swap the values. 

FINAL RULE

Accuracy is more important than completeness.

It is always preferable to return fewer fields with high confidence than many uncertain fields.
"""
)

In [87]:
# ---------------------------------------------------------------------------
# Deterministic parsing of the split documents
#
# Only documents that passed the extraction gate are parsed. `should_extract`
# is the gate itself (classified as an invoice-like Original), as opposed to
# `invoice_data is not None`, which would also drop documents whose extraction
# call failed for transient reasons.
# ---------------------------------------------------------------------------


from invoice_utils.pdf_parser import build_attachment_evidence

evidences: dict[str, dict] = {}

for doc, cls, invoice_data in results:
    if not should_extract(cls):
        state = cls.document_state.value if cls.document_state is not None else "no state"
        print(f"⏭️  Not parsing {doc.filename}: {cls.document_type.value} / {state}")
        continue

    evidences[doc.filename] = build_attachment_evidence(
        split_paths[doc.filename],
        max_pages_text=10,
        max_pages_vision=4,
        dpi=100,
        min_good_chars_per_page=200,
        min_font_size=5,
    )
    print(f"✅ Parsed {doc.filename}")

print(f"\n{len(evidences)} document(s) parsed.")

for filename, ev in evidences.items():
    print("=" * 80)
    print(
        f"📄 {filename}  "
        f"good_score={ev['good_score']}  text_only={ev['allow_text_only']}  "
    )
    print("=" * 80)
    print(ev["extracted_text"])
    print()

✅ Parsed Invoice 229555.pdf

1 document(s) parsed.
📄 Invoice 229555.pdf  good_score=553  text_only=True  
PAGE	DOC.	DATE
																					1	02/07/26
																					Invoice	no:
														INVOICE							229555
GSLines	Transportes		Maritimos		LDA
Avenida	do Mar	e das	Comunidades									V.A.T.	-NUMBER				Our	ref.			Due date
madeirenses		21 2nd										PT511011911					2628020				01/08/2026
9000-054
Portugal
												Your	ref.
												5000355882
Vessel											Voyage			DATE				B/L
WEC	DE HOOGH										EH612A			05/04/2026
NUMBER	& TYPE	OF	CONTAINERS			WEIGHT												OUR BOOKING		NUMBER

P.O.O.						POL						POD						FINAL	DESTINATION
						ANTWERPEN						LEIXOES
Description							Curr.			Base			Rate						Exchange		Total	EUR	Vat
REMUNERATION		EXPORT		CARGO			EUR			2,000		52,000									104,00

VAT			VAT	description				Basis	VAT		% VAT	Amount				please	mention		on your	payment
NO BTW								104,00		0,00		0,00
														Custumor	NR.		INVOICE		NR		TO	PAY EUR
						

In [88]:
# ---------------------------------------------------------------------------
# Node 4 — Validation (one LLM call per extracted document)
#
# Same loop shape as the parsing node: iterate `results` and only validate the
# documents that were actually extracted. `evidences` is keyed by filename and
# only holds the gated-in documents, so a missing entry means "no parsed text"
# rather than an error.
# ---------------------------------------------------------------------------

llm_with_tools = llm.bind_tools([verify_client_nif, verify_supplier_nif])

validations: dict[str, str] = {}

for doc, cls, invoice_data in results:
    if invoice_data is None:
        state = cls.document_state.value if cls.document_state is not None else "no state"
        print(f"⏭️  Not validating {doc.filename}: {cls.document_type.value} / {state}")
        continue

    parsed_text = evidences.get(doc.filename, {}).get("extracted_text")

    question = build_validation_human_message(invoice_data, parsed_text=parsed_text)
    try:
        answer = ask_with_tools(
            question,
            llm_with_tools,
            [verify_client_nif, verify_supplier_nif],
            system_message=validation_system_message,
        )
    except Exception as exc:  # noqa: BLE001 - keep the batch alive, inspect after
        print(f"❌ Failed to validate {doc.filename}: {type(exc).__name__}: {exc}")
        continue

    validations[doc.filename] = answer
    print(f"✅ Validated {doc.filename}")

print(f"\n{len(validations)} document(s) validated.")

for filename, answer in validations.items():
    print("=" * 80)
    print(f"📄 {filename}")
    print("=" * 80)
    print(answer)
    print()


✅ Validated Invoice 229555.pdf

1 document(s) validated.
📄 Invoice 229555.pdf
The extracted data is mostly valid, but there are a few discrepancies and clarifications needed:

1. **Supplier Name**: "GSLines Transportes Maritimos LDA"
   - **Validity**: Valid
   - **Evidence**: The name appears in the parsed text.

2. **Supplier VAT**: "NL813715532B01"
   - **Validity**: Invalid
   - **Evidence**: The supplier VAT number does not match any known supplier in our records. The client VAT "PT511011911" is found in the parsed text and matches a known client, indicating a possible mix-up.

3. **Client Name**: "RateBase Exchange Total EUR"
   - **Validity**: Invalid
   - **Evidence**: The extracted client name does not appear in the parsed text. The correct client name is likely missing or misinterpreted.

4. **Client VAT**: "PT511011911"
   - **Validity**: Valid
   - **Evidence**: The client VAT number appears in the parsed text and matches a known client.

5. **Purchase Order**: "2628020"
  